# Use context managers

This guide shows you how to use context managers and the `with` statement to manage resources safely. You will learn how to use built-in context managers, create your own, and handle exceptions within them.

**What you need to know:** Basic exception handling with `try`/`except`/`finally`, and familiarity with Python classes.

## The problem

When your code acquires a resource (a file, a network connection, a database cursor, or a lock), you must release it when you are finished. If an exception occurs before the cleanup code runs, the resource leaks. Writing `try`/`finally` blocks for every resource is verbose, repetitive, and easy to get wrong.

Context managers solve this problem. They guarantee that setup and cleanup code always runs, even when exceptions are raised.

## Using built-in context managers

The most common context manager in Python is the built-in `open()` function for file handling. Compare the manual approach with the context manager approach.

In [ ]:
import tempfile
import os

# Create a temporary file for demonstrations throughout this guide
demo_dir = tempfile.mkdtemp()
demo_path = os.path.join(demo_dir, "example.txt")

with open(demo_path, "w", encoding="utf-8") as f:
    f.write("First line\nSecond line\nThird line\n")

print(f"Created demo file: {demo_path}")

In [ ]:
# Manual approach: verbose and error-prone
f = open(demo_path, "r", encoding="utf-8")
try:
    content = f.read()
finally:
    f.close()

print(f"Manual approach: read {len(content)} characters")
print(f"File closed: {f.closed}")

In [ ]:
# Context manager approach: clean and safe
with open(demo_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"Context manager approach: read {len(content)} characters")
print(f"File closed: {f.closed}")

Both examples achieve the same result, but the `with` statement is shorter, clearer, and eliminates the risk of forgetting the `finally` block. When the `with` block ends, whether normally or because of an exception, Python automatically closes the file.

## How context managers work

A context manager is any object that implements two special methods:

- `__enter__()` is called when execution enters the `with` block. Its return value is bound to the variable after `as`.
- `__exit__(exc_type, exc_val, exc_tb)` is called when execution leaves the `with` block. It receives information about any exception that occurred (or `None` values if no exception was raised).

Here is what happens step by step:

1. Python calls `__enter__()` on the context manager object.
2. The return value is assigned to the variable after `as`.
3. The body of the `with` block executes.
4. Python calls `__exit__()`, passing exception information if an exception occurred.
5. If `__exit__()` returns `True`, the exception is suppressed. Otherwise, it propagates.

The following example makes this visible.

In [ ]:
class DemoContextManager:
    """A simple context manager that prints when it enters and exits."""

    def __enter__(self) -> str:
        """Set up the resource and return it."""
        print("__enter__ called: acquiring resource")
        return "the managed resource"

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: object | None,
    ) -> bool:
        """Clean up the resource."""
        print(f"__exit__ called: releasing resource")
        print(f"  Exception type: {exc_type}")
        print(f"  Exception value: {exc_val}")
        # Return False to let any exception propagate
        return False


print("--- Normal execution ---")
with DemoContextManager() as resource:
    print(f"Inside with block, resource = {resource!r}")

print()

In [ ]:
print("--- Execution with an exception ---")
try:
    with DemoContextManager() as resource:
        print(f"Inside with block, resource = {resource!r}")
        raise ValueError("Something went wrong")
except ValueError as e:
    print(f"Exception handled outside with block: {e}")

Notice that `__exit__()` was called in both cases. When an exception occurred, the exception details were passed to `__exit__()`. Because `__exit__()` returned `False`, the exception was not suppressed and propagated to the outer `try`/`except`.

## Creating a custom context manager class

To create your own context manager, define a class with `__enter__()` and `__exit__()` methods. The following example implements a managed database connection using `sqlite3`.

In [ ]:
import sqlite3


class DatabaseConnection:
    """A context manager for SQLite database connections.

    Automatically commits the transaction on success
    or rolls it back if an exception occurs.
    """

    def __init__(self, db_path: str) -> None:
        self.db_path = db_path
        self.connection: sqlite3.Connection | None = None

    def __enter__(self) -> sqlite3.Connection:
        """Open the database connection."""
        self.connection = sqlite3.connect(self.db_path)
        return self.connection

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: object | None,
    ) -> bool:
        """Close the connection, committing or rolling back as needed."""
        if self.connection is not None:
            if exc_type is not None:
                # An exception occurred: roll back the transaction
                self.connection.rollback()
                print("Transaction rolled back")
            else:
                # No exception: commit the transaction
                self.connection.commit()
                print("Transaction committed")
            self.connection.close()
            print("Connection closed")
        # Return False to propagate any exception
        return False

In [ ]:
db_path = os.path.join(demo_dir, "example.db")

# Successful transaction
with DatabaseConnection(db_path) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT)")
    conn.execute("INSERT INTO users (name) VALUES (?)", ("Alice",))
    conn.execute("INSERT INTO users (name) VALUES (?)", ("Bob",))

# Verify the data was committed
with DatabaseConnection(db_path) as conn:
    cursor = conn.execute("SELECT name FROM users ORDER BY name")
    names = [row[0] for row in cursor]
    print(f"Users in database: {names}")

In [ ]:
# Failed transaction: the insertion is rolled back
try:
    with DatabaseConnection(db_path) as conn:
        conn.execute("INSERT INTO users (name) VALUES (?)", ("Charlie",))
        raise RuntimeError("Simulated failure after insert")
except RuntimeError as e:
    print(f"Handled exception: {e}")

# Verify that Charlie was not added
with DatabaseConnection(db_path) as conn:
    cursor = conn.execute("SELECT name FROM users ORDER BY name")
    names = [row[0] for row in cursor]
    print(f"Users in database after rollback: {names}")

The `DatabaseConnection` context manager ensures that the connection is always closed and that transactions are committed on success or rolled back on failure. You do not need to remember to call `commit()`, `rollback()`, or `close()` yourself.

## Using the `contextlib.contextmanager` decorator

Writing a class with `__enter__()` and `__exit__()` can be verbose for simple cases. The `contextlib` module provides the `@contextmanager` decorator, which lets you write a context manager as a generator function.

The pattern is as follows:

1. Perform setup before the `yield` statement.
2. `yield` the resource (this is the value bound to the `as` variable).
3. Perform cleanup after the `yield` statement.

Wrap the `yield` in a `try`/`finally` block to guarantee that cleanup runs even if an exception is raised inside the `with` block.

In [ ]:
from contextlib import contextmanager
from typing import Generator


@contextmanager
def managed_file(path: str, mode: str = "r") -> Generator[object, None, None]:
    """Open a file and ensure it is closed when the block exits.

    Args:
        path: The path to the file.
        mode: The file mode (for example, 'r', 'w', 'a').

    Yields:
        The open file object.
    """
    f = open(path, mode, encoding="utf-8")
    try:
        print(f"Opening {path!r} in mode {mode!r}")
        yield f
    finally:
        f.close()
        print(f"Closed {path!r}")


with managed_file(demo_path) as f:
    content = f.read()
    print(f"Read {len(content)} characters")

The `@contextmanager` decorator is especially useful when you need a quick, lightweight context manager and do not need the full flexibility of a class.

### Alternative: compare the two approaches

| Approach | Best for | Advantages |
|----------|----------|------------|
| Class with `__enter__`/`__exit__` | Complex resource management, stateful context managers | Full control over exception handling, reusable, can store state |
| `@contextmanager` decorator | Simple setup/cleanup patterns | Less boilerplate, easier to read, faster to write |

## Handling exceptions inside context managers

A context manager can choose to suppress an exception by returning `True` from `__exit__()`. This is useful when you want the context manager to handle certain exceptions silently.

> **Important:** Use exception suppression sparingly. Silently swallowing exceptions can hide bugs. Only suppress exceptions that you expect and know how to handle.

In [ ]:
class SuppressSpecificException:
    """A context manager that suppresses a specific exception type.

    Any other exception types propagate normally.
    """

    def __init__(self, *exception_types: type[BaseException]) -> None:
        self.exception_types = exception_types

    def __enter__(self) -> "SuppressSpecificException":
        """Enter the context."""
        return self

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: object | None,
    ) -> bool:
        """Suppress the exception if it matches the configured types."""
        if exc_type is not None and issubclass(exc_type, self.exception_types):
            print(f"Suppressed {exc_type.__name__}: {exc_val}")
            return True
        return False


# The ValueError is suppressed
with SuppressSpecificException(ValueError):
    print("About to raise a ValueError")
    raise ValueError("This will be suppressed")

print("Execution continues normally after the with block")

In [ ]:
# A TypeError is NOT suppressed because it is not in the configured types
try:
    with SuppressSpecificException(ValueError):
        raise TypeError("This will propagate")
except TypeError as e:
    print(f"TypeError was not suppressed: {e}")

Python provides a built-in version of this pattern in the `contextlib` module: `contextlib.suppress()`.

In [ ]:
from contextlib import suppress

# Attempt to delete a file that may not exist
nonexistent_path = os.path.join(demo_dir, "does_not_exist.txt")

with suppress(FileNotFoundError):
    os.remove(nonexistent_path)

print("No exception raised, even though the file did not exist")

### Handling exceptions in `@contextmanager` functions

When using the `@contextmanager` decorator, you can handle exceptions around the `yield` statement. The exception raised inside the `with` block is re-raised at the point of the `yield`.

In [ ]:
@contextmanager
def error_logging_context(label: str) -> Generator[None, None, None]:
    """Log any exception that occurs within the block, then re-raise it.

    Args:
        label: A label to identify the context in log messages.

    Yields:
        Nothing.
    """
    print(f"[{label}] Entering context")
    try:
        yield
    except Exception as e:
        print(f"[{label}] Exception logged: {type(e).__name__}: {e}")
        raise
    finally:
        print(f"[{label}] Exiting context")


# The exception is logged and then re-raised
try:
    with error_logging_context("data-processing"):
        result = int("not_a_number")
except ValueError:
    print("ValueError handled outside the context manager")

If you want the `@contextmanager` to suppress the exception instead of re-raising it, simply do not include the `raise` statement in the `except` block. However, be cautious with this approach, as suppressing exceptions can make debugging difficult.

## Multiple context managers in a single `with` statement

You can use multiple context managers in a single `with` statement by separating them with commas. All resources are guaranteed to be cleaned up, even if an exception occurs.

In [ ]:
# Create a second file for the copy demonstration
source_path = demo_path
dest_path = os.path.join(demo_dir, "copy.txt")

# Open two files simultaneously using a single with statement
with (
    open(source_path, "r", encoding="utf-8") as source,
    open(dest_path, "w", encoding="utf-8") as dest,
):
    content = source.read()
    dest.write(content.upper())

# Verify both files are closed
print(f"Source file closed: {source.closed}")
print(f"Destination file closed: {dest.closed}")

# Check the result
with open(dest_path, "r", encoding="utf-8") as f:
    print(f"Copied content (uppercased):\n{f.read()}")

The parenthesised form shown above (available from Python 3.10 onwards) is the recommended style when the line would otherwise be too long. For shorter expressions, you can write them on a single line:

```python
with open("a.txt") as a, open("b.txt") as b:
    ...
```

When any of the context managers raises an exception during `__enter__()`, the context managers that have already entered are exited in reverse order.

## Practical example: a timer context manager

The following example demonstrates a reusable `Timer` context manager that measures how long a block of code takes to execute. This is useful for profiling and performance measurement.

In [ ]:
import time


class Timer:
    """A context manager that measures the elapsed time of a code block.

    Attributes:
        label: A descriptive label for the timed operation.
        elapsed: The elapsed time in seconds (set after the block completes).
    """

    def __init__(self, label: str = "Operation") -> None:
        self.label = label
        self.elapsed: float = 0.0
        self._start_time: float = 0.0

    def __enter__(self) -> "Timer":
        """Record the start time."""
        self._start_time = time.perf_counter()
        return self

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: object | None,
    ) -> bool:
        """Calculate the elapsed time and print the result."""
        self.elapsed = time.perf_counter() - self._start_time
        print(f"[{self.label}] Elapsed time: {self.elapsed:.4f} seconds")
        # Do not suppress exceptions
        return False

In [ ]:
with Timer("Sum calculation") as t:
    total = sum(range(1_000_000))

print(f"Total: {total}")
print(f"Elapsed time available as attribute: {t.elapsed:.4f} seconds")

In [ ]:
# The timer still records elapsed time even when an exception occurs
try:
    with Timer("Failing operation") as t:
        time.sleep(0.1)
        raise RuntimeError("Something went wrong")
except RuntimeError:
    print(f"Exception occurred after {t.elapsed:.4f} seconds")

Because `__exit__()` does not suppress the exception (it returns `False`), the `RuntimeError` propagates normally. Yet the elapsed time is still recorded, which is useful for debugging performance issues even when operations fail.

### Alternative: timer using `@contextmanager`

The same timer can be written more concisely with the `@contextmanager` decorator. The trade-off is that you cannot easily store the elapsed time as an attribute.

In [ ]:
@contextmanager
def timer(label: str = "Operation") -> Generator[None, None, None]:
    """Measure and print the elapsed time of a code block.

    Args:
        label: A descriptive label for the timed operation.

    Yields:
        Nothing.
    """
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] Elapsed time: {elapsed:.4f} seconds")


with timer("List comprehension"):
    squares = [x ** 2 for x in range(1_000_000)]

## Practical example: managed temporary directory

Another common use case is creating a temporary directory that is automatically cleaned up when the `with` block exits. Python provides `tempfile.TemporaryDirectory` for this, but building your own demonstrates the pattern.

In [ ]:
import shutil


@contextmanager
def managed_temp_directory(prefix: str = "tmp_") -> Generator[str, None, None]:
    """Create a temporary directory that is removed when the block exits.

    Args:
        prefix: The prefix for the temporary directory name.

    Yields:
        The path to the temporary directory.
    """
    temp_dir = tempfile.mkdtemp(prefix=prefix)
    print(f"Created temporary directory: {temp_dir}")
    try:
        yield temp_dir
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)
        print(f"Removed temporary directory: {temp_dir}")


with managed_temp_directory(prefix="demo_") as tmp:
    # Create some files in the temporary directory
    for i in range(3):
        filepath = os.path.join(tmp, f"file_{i}.txt")
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"Content of file {i}")
    print(f"Files in temp dir: {os.listdir(tmp)}")

# Verify the directory was removed
print(f"Directory still exists: {os.path.exists(tmp)}")

> **Tip:** In production code, use `tempfile.TemporaryDirectory()` from the standard library, which provides the same behaviour with additional safety features.

## Cleanup

Remove the temporary files and directories created during this guide.

In [ ]:
shutil.rmtree(demo_dir, ignore_errors=True)
print(f"Cleaned up demo directory: {demo_dir}")

## Summary

This guide covered the following techniques:

- **Built-in context managers** such as `open()` provide automatic resource cleanup with the `with` statement.
- **`__enter__()` and `__exit__()`** are the two methods that make a class work as a context manager.
- **Custom context manager classes** give you full control over setup, cleanup, and exception handling.
- **The `@contextmanager` decorator** lets you write simple context managers as generator functions with less boilerplate.
- **Exception suppression** is possible by returning `True` from `__exit__()`, but should be used sparingly.
- **Multiple context managers** can be combined in a single `with` statement using commas or parentheses.
- **Practical patterns** such as timers and managed temporary directories demonstrate how context managers simplify real-world code.